In [ ]:
%%capture
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langchain-upstage>=0.3.0" "langchain-community>=0.3.0" "langgraph>=0.2.0" "langchain-text-splitters>=0.3.0" "chromadb>=0.5.0" "tiktoken>=0.7.0" "pymupdf>=1.24.0" "python-dotenv>=1.0.0" -q 

In [ ]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY를 입력하세요: ")

if not os.environ.get("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = input("🔑 UPSTAGE_API_KEY를 입력하세요: ")
print("✅ 환경 설정 완료") 

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)
# ═══════════════════════════════════════════════════════════════

# ── Option A: OpenAI (기본) ──
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage, UpstageEmbeddings
llm = ChatUpstage(model="solar-pro3")
embeddings = UpstageEmbeddings(model="embedding-query") 

In [ ]:
# 모델 연결 테스트
response = llm.invoke("Yes24 배송 정책에 대해 알려줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})") 

## 문맥 추가하기

In [ ]:
# 데이터 파일 경로 설정
# 실습 폴더 내 data/ 디렉토리에 PDF 파일이 위치해야 합니다.
# 예시 디렉토리 구조:
#   Practice/
#   ├── (실습-문제) 4-1_RAG_기반_Customer_Service_AI_에이전트_개발.ipynb  ← 현재 파일
#   └── data/
#       ├── 총알배송_서비스 혜택 - 예스24.pdf
#       ├── 배송지연 보상제도_서비스 혜택 - 예스24.pdf
#       └── ...
DATA_DIR = "data/"

# 데이터 파일 목록 확인
import glob
import os

pdf_files = glob.glob(DATA_DIR + "*.pdf")
print(f"📁 발견된 PDF 파일: {len(pdf_files)}개")
for f in pdf_files:
    print(f"  - {os.path.basename(f)}")

if len(pdf_files) == 0:
    print(f"\n⚠️ PDF 파일이 발견되지 않았습니다.")
    print(f"   '{DATA_DIR}' 경로에 PDF 파일을 배치해주세요.")

In [ ]:
# TODO 2: 자료를 직접 프롬프트에 넣어 질문하기
from langchain_community.document_loaders import PyMuPDFLoader

# 총알배송 문서 로드
pdf_path = DATA_DIR + "총알배송_서비스 혜택 - 예스24.pdf"
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

# 문서 내용 확인
doc_content = "\n".join([doc.page_content for doc in documents])
print(f"📄 문서 길이: {len(doc_content)} 글자")
print(f"\n📄 문서 내용 (일부):\n{doc_content[:500]}...") 

In [ ]:
# 컨텍스트를 포함한 프롬프트 구성
from langchain_core.prompts import ChatPromptTemplate

# ========== 여기에 코드를 작성하세요 ==========
# ChatPromptTemplate을 사용하여 프롬프트 템플릿을 만드세요
# 힌트: system 메시지에 {context} 플레이스홀더를 포함하고,
#       human 메시지에 {question} 플레이스홀더를 포함

prompt_template = ChatPromptTemplate.from_messages([
    ('system', """
    당신은 AI 온라인 서점의 고객 서비스 상담원입니다.
    다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
    자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다"라고 답변하세요.
    
    [참고 자료]
    {context}
    """),
    ('human', '{question}'),
])

# ============================================

if prompt_template:
    # 질문
    question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

    # 프롬프트 생성 후 LLM 직접 호출
    messages = prompt_template.format_messages(context=doc_content, question=question)
    response = llm.invoke(messages)

    print("📝 질문:", question)
    print("\n🤖 자료 기반 응답:")
    print(response.content)
else:
    print("⚠️ prompt_template을 완성해주세요!") 

In [ ]:
# 키워드 검색 예시
if "총알배송" in document:
    return document


🔍 키워드 검색 테스트:
  '빠른 배송': 0개 문서 발견
  '배송 빨리': 0개 문서 발견
  '신속 배송': 0개 문서 발견
  '총알배송': 8개 문서 발견

## 의미 기반 검색 + DB화 기초

### 임베딩

In [ ]:
# TODO: 문서를 Vector DB에 저장하기
from langchain_community.vectorstores import Chroma
import tiktoken


all_documents = []
for pdf_path in pdf_files:
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    # 파일명을 메타데이터에 추가
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

print(f"📚 총 {len(all_documents)}개 페이지 로드 완료")


# 임베딩 모델의 토큰 제한 처리 (text-embedding-3-small: 최대 8,191 토큰)
# PDF 전체 페이지는 토큰 제한을 초과할 수 있으므로 사전에 잘라줌
MAX_TOKENS = 8000  # 안전 여유분 확보
enc = tiktoken.encoding_for_model("text-embedding-3-small")

truncated_count = 0
for doc in all_documents:
    tokens = enc.encode(doc.page_content)
    if len(tokens) > MAX_TOKENS:
        doc.page_content = enc.decode(tokens[:MAX_TOKENS])
        truncated_count += 1

if truncated_count > 0:
    print(f"⚠️ {truncated_count}개 문서가 토큰 제한 초과로 잘림 (최대 {MAX_TOKENS} 토큰)")

# ========== 여기에 코드를 작성하세요 ==========
# Vector Store 생성 (문서 단위 - 페이지별)
# 힌트: Chroma.from_documents(documents=..., embedding=..., collection_name=...)

vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name='yes24_docs_page',
)

# ============================================

if vectorstore:
    print(f"✅ Vector Store 생성 완료")
    print(f"📊 저장된 문서 수: {vectorstore._collection.count()}개")
else:
    print("⚠️ vectorstore를 완성해주세요!") 

In [ ]:
# TODO: 의미 기반 검색 수행하기

# ========== 여기에 코드를 작성하세요 ==========
# Retriever 생성
# 힌트: vectorstore.as_retriever(search_kwargs={"k": 3})

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# ============================================

if retriever:
    # 의미 기반 검색 테스트
    test_queries = [
        "빠른 배송",      # 총알배송과 유사한 의미
        "배송 빨리",      # 총알배송과 유사한 의미  
        "신속 배송",      # 총알배송과 유사한 의미
        "총알배송",       # 정확한 키워드
    ]

    print("🔍 의미 기반 검색 테스트:")
    for query in test_queries:
        results = retriever.invoke(query)
        source_files = set([doc.metadata.get('source_file', 'Unknown') for doc in results])
        print(f"  '{query}': {source_files}")
else:
    print("⚠️ retriever를 완성해주세요!") 

In [ ]:
# 검색 결과 상세 확인
if retriever:
    query = "배송이 늦으면 어떻게 되나요?"
    results = retriever.invoke(query)

    print(f"🔍 질문: {query}")
    print(f"\n📄 검색 결과 ({len(results)}개):")
    for i, doc in enumerate(results, 1):
        print(f"\n--- 결과 {i} ---")
        print(f"출처: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"내용: {doc.page_content[:200]}...") 

## 청킹(Chunking) 전략

In [ ]:
# TODO: 청킹 전략 비교하기
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 다양한 청킹 설정 비교
chunking_configs = [
    {"chunk_size": 100, "chunk_overlap": 0},
    {"chunk_size": 200, "chunk_overlap": 20},
    {"chunk_size": 500, "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 100},
]

# 하나의 문서로 테스트
test_doc = all_documents[0]
print(f"📄 원본 문서 길이: {len(test_doc.page_content)} 글자\n")

for config in chunking_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        length_function=len,
    )
    chunks = splitter.split_documents([test_doc])
    print(f"chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']}: {len(chunks)}개 청크") 

In [ ]:
# ========== 여기에 코드를 작성하세요 ==========
# 최적 청킹 설정을 선택하고 전체 문서를 청킹하세요
# 권장 설정: chunk_size=300, chunk_overlap=50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', ''],
)

# ============================================

if text_splitter:
    # 전체 문서 청킹
    chunked_documents = text_splitter.split_documents(all_documents)

    print(f"📊 청킹 결과:")
    print(f"  - 원본 문서 수: {len(all_documents)}개")
    print(f"  - 청킹 후 조각 수: {len(chunked_documents)}개")
    print(f"  - 평균 청크 크기: {sum(len(doc.page_content) for doc in chunked_documents) / len(chunked_documents):.0f}자")
else:
    print("⚠️ text_splitter를 완성해주세요!") 

In [ ]:
# 청킹된 문서로 새 Vector Store 생성
if text_splitter and 'chunked_documents' in dir():
    vectorstore_chunked = Chroma.from_documents(
        documents=chunked_documents,
        embedding=embeddings,
        collection_name="yes24_docs_chunked"
    )

    retriever_chunked = vectorstore_chunked.as_retriever(search_kwargs={"k": 3})

    print(f"✅ 청킹된 문서로 Vector Store 재생성 완료")
    print(f"📊 저장된 청크 수: {vectorstore_chunked._collection.count()}개") 

In [ ]:
# 청킹 전후 검색 품질 비교
if retriever and 'retriever_chunked' in dir():
    query = "배송이 늦으면 보상받을 수 있나요?"

    print(f"🔍 질문: {query}\n")

    # 페이지 단위 검색
    results_page = retriever.invoke(query)
    print("📄 [페이지 단위 검색]")
    print(f"  - 첫 번째 결과 길이: {len(results_page[0].page_content)}자")
    print(f"  - 내용: {results_page[0].page_content[:150]}...")

    print()

    # 청크 단위 검색
    results_chunk = retriever_chunked.invoke(query)
    print("📄 [청크 단위 검색]")
    print(f"  - 첫 번째 결과 길이: {len(results_chunk[0].page_content)}자")
    print(f"  - 내용: {results_chunk[0].page_content[:150]}...") 

## LangGraph

In [ ]:
# TODO: LangGraph StateGraph로 RAG 파이프라인 구현
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document

# 1. State 타입 정의
# - question: 사용자 질문
# - context: 검색된 문서 내용
# - answer: 생성된 답변
class RAGState(TypedDict):
    question: str
    context: str
    answer: str

# ========== 여기에 코드를 작성하세요 ==========

# 2. retrieve 노드: 질문으로 관련 문서 검색
# 힌트: retriever_chunked.invoke(question)을 사용하여 문서를 검색하고,
#       검색 결과를 문자열로 포맷팅하여 {"context": ...} 형태로 반환
def retrieve(state: RAGState) -> RAGState:
    """Vector Store에서 관련 문서를 검색하는 노드"""
    question = state['question']
    docs = retriever_chunked.invoke(question)
    context = '\n\n'.join([doc.page_content for doc in docs])
    return {'context': context}

# 3. generate 노드: 검색된 문서로 답변 생성
# 힌트: state에서 question과 context를 가져와서
#       ChatPromptTemplate으로 프롬프트를 생성하고
#       llm.invoke(messages)로 답변을 생성한 후
#       {"answer": response.content} 형태로 반환
def generate(state: RAGState) -> RAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드"""
    question = state['question']
    context = state['context']

    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", """
            당신은 AI 온라인 서점 'Yes24'의 고객 서비스 상담원입니다.
            다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.
            
            [검색된 자료]
            {context}
            
            답변 규칙:
            1. 검색된 자료를 기반으로 답변하세요
            2. 자료에 없는 내용은 추측하지 마세요
            3. 존댓말을 사용하세요
            4. 간결하고 명확하게 답변하세요
        """),
        ("human", "{question}")
    ])

    messages = rag_prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    answer = response.content
    return {'answer': answer}

# 4. StateGraph 구성
# - 노드 추가: workflow.add_node("retrieve", retrieve)
# - 엣지 연결: workflow.add_edge(START, "retrieve")
# - 그래프 컴파일: workflow.compile()

workflow = StateGraph(RAGState)

# 노드 추가
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)

# 엣지 연결
workflow.add_edge(START, 'retrieve')
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)

rag_graph = workflow.compile()

# ============================================

if rag_graph:
    print("✅ LangGraph RAG 파이프라인 구성 완료!")
else:
    print("⚠️ rag_graph를 완성해주세요!") 

In [ ]:
# LangGraph RAG 파이프라인 테스트
question = "Yes24에서 총알배송이 뭔가요? 어떤 조건에서 가능한가요?"

print("📝 질문:", question)
print("\n🤖 LangGraph RAG 응답:")

# 그래프 실행
result = rag_graph.invoke({"question": question})
print(result["answer"]) 

In [ ]:
# 다양한 질문 테스트
if rag_graph:
    test_questions = [
        "배송이 늦으면 어떻게 보상받을 수 있나요?",
        "YES포인트는 어떻게 적립되나요?",
        "신규 회원 혜택은 무엇인가요?",
        "도서가 품절되면 어떻게 되나요?"
    ]

    for q in test_questions:
        print(f"\n{'='*50}")
        print(f"📝 질문: {q}")
        print(f"\n🤖 LangGraph RAG 응답:")
        result = rag_graph.invoke({"question": q})
        print(result["answer"]) 

In [ ]:
from langchain_core.messages import HumanMessage

# Step 2 vs Step 7 비교
if rag_graph:
    comparison_question = "Yes24 배송지연 보상제도는 어떻게 되나요?"

    print("="*60)
    print("📊 Step 2 vs Step 7 비교")
    print("="*60)
    print(f"\n📝 질문: {comparison_question}")

    # Step 2: LLM만
    print("\n[Step 2: LLM만 사용]")
    response_llm_only = llm.invoke([HumanMessage(content=comparison_question)])
    print(response_llm_only.content)

    # Step 7: LangGraph RAG
    print("\n[Step 7: LangGraph RAG 사용]")
    result = rag_graph.invoke({"question": comparison_question})
    print(result["answer"]) 